# 조업 이벤트 데이터 분석 및 정리

**담당: 김정렬**  ·  관련 Issue: #28

목표

행 수, 컬럼(EVT_ID, EVT_DT, EVT_TYPE, PLANT_CD, BEF_VAL, AFT_VAL), 결측·중복 확인

EVT_TYPE 종류별 건수 세기

PLANT_CD 종류별 건수 세기

PLANT_CD와 EVT_TYPE을 같이 놓고 어떤 설비에 어떤 이벤트가 있는지 표로 보기

이벤트 종류별로 BEF_VAL, AFT_VAL에 어떤 값이 들어 있는지 몇 개씩 보기

월별 이벤트 건수 세기

노트북 맨 아래에 결과 정리 표 채우기

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
TEMP_CSV   = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV  = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")

pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

In [ ]:
# 조업이벤트 데이터 불러오기
event_df = pd.read_csv(EVENT_CSV)

### 1. 행 수, 컬럼(EVT_ID, EVT_DT, EVT_TYPE, PLANT_CD, BEF_VAL, AFT_VAL), 결측·중복 확인

In [ ]:
# 분석에 사용할 주요 컬럼
event_cols = [
    "EVT_ID",
    "EVT_DT",
    "EVT_TYPE",
    "PLANT_CD",
    "BEF_VAL",
    "AFT_VAL"
]

print("전체 행 수:", len(event_df))
print("전체 컬럼 수:", len(event_df.columns))

print("\n[주요 컬럼]")
print(event_cols)

print("\n[결측치 개수]")
display(event_df[event_cols].isnull().sum())

print("\n[중복 행 개수]")
print(event_df.duplicated().sum())

### 2. EVT_TYPE 종류별 건수 세기

In [ ]:
# EVT_TYPE 종류별 건수
evt_type_counts = (
    event_df["EVT_TYPE"]
    .value_counts(dropna=False)
    .rename_axis("EVT_TYPE")
    .reset_index(name="COUNT")
)

print("EVT_TYPE 종류 수:", event_df["EVT_TYPE"].nunique())

display(evt_type_counts)

### 3. PLANT_CD 종류별 건수 세기

In [ ]:
# PLANT_CD 종류별 건수
plant_counts = (
    event_df["PLANT_CD"]
    .value_counts(dropna=False)
    .rename_axis("PLANT_CD")
    .reset_index(name="COUNT")
)

print("PLANT_CD 종류 수:", event_df["PLANT_CD"].nunique())

display(plant_counts)


### 4. PLANT_CD와 EVT_TYPE을 같이 놓고 어떤 설비에 어떤 이벤트가 있는지 표로 보기

In [ ]:
# PLANT_CD별 EVT_TYPE 발생 건수
plant_event_table = pd.crosstab(
    event_df["PLANT_CD"],
    event_df["EVT_TYPE"]
)

display(plant_event_table)

### 5. 이벤트 종류별로 BEF_VAL, AFT_VAL에 어떤 값이 들어 있는지 몇 개씩 보기

In [ ]:
# EVT_TYPE별 BEF_VAL, AFT_VAL 샘플 확인
event_value_samples = (
    event_df[
        ["EVT_TYPE", "BEF_VAL", "AFT_VAL"]
    ]
    .groupby("EVT_TYPE", group_keys=False)
    .head()
)

display(event_value_samples)

### 6. 월별 이벤트 건수 세기

In [ ]:
# EVT_DT 날짜형으로 변환
event_df["EVT_DT"] = pd.to_datetime(
    event_df["EVT_DT"],
    errors="coerce"
)

# 월 정보 생성
event_df["YEAR_MONTH"] = event_df["EVT_DT"].dt.to_period("M")

# 월별 이벤트 건수
monthly_counts = (
    event_df["YEAR_MONTH"]
    .value_counts()
    .sort_index()
    .rename_axis("YEAR_MONTH")
    .reset_index(name="COUNT")
)

display(monthly_counts)

### 결과정리

| 항목 | 확인한 값 |
| --- | --- |
| 전체 행 수 | 263 행 |
| 전체 결측치 수 | 0 개 |
| 전체 중복 행 수 | 0 행 |
| EVT_TYPE 종류 수 | 3 개 |
| PLANT_CD 종류 수 | 5 개 |

PLANT_CD별 EVT_TYPE 발생 건수
| EVT_TYPE | 강종변경 | 원료배합변경 | 조업률변경 |
| --- | --- | --- | --- |
| BF1 | 0 | 13 | 34 |
| BF2 | 0 | 7 | 34 |
| CR1 | 134 | 0 | 0 |
| SP1 | 0 | 0 | 23 |
| SP2 | 0 | 0 | 18 |

### 알 수 있는 것

1. 조업이벤트 데이터는 결측치, 중복 행이 0개라서 바로 활용하기 좋습니다.

2. 조업이벤트 데이터에서 CR1은 강종변경 이벤트만 확인됩니다.

따라서 CR1에서만 연속소둔로를 다룬다면 다른 PLANT_CD는 제외가 가능합니다.

### 확실하지 않은 점

조업이벤트 파일만으로는 CR1의 134건의 강종변경이 온도 변화에 연결이 되는지 모릅니다.

이를 확인하려면 EVT_DT를 기준으로 온도 데이터의 TC·OP 변화와 대조해야 합니다.